# ML Zoomcamp 2026 — Homework 1

Dataset: 2026 Car Fuel Efficiency.

**Rule for this notebook:** type every line yourself. Don't paste.
Each question tells you the *concept* and the *function* — the code is yours to write.

Submit answers at: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw01

## Setup

In [44]:
import pandas as pd
import numpy as np

pd.__version__

'2.2.3'

### Q1. Pandas version

Already answered by the cell above. Note the value and move on.

## Getting the data

Download the CSV into this folder. Run this once (the `!` sends the line to the shell):

In [2]:
!curl -o car_fuel_efficiency_2026.csv https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0  620k    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  620k  100  620k    0     0   578k      0  0:00:01  0:00:01 --:--:--  580k


In [45]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


Before answering anything, look at the shape of the thing you loaded.
`df.info()` is the single most useful first command in pandas: it shows every column,
its dtype, and how many **non-null** values it has. Most of Q2 and Q4 are visible here.

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   model_year           10000 non-null  int64  
 1   origin               10000 non-null  object 
 2   fuel_type            10000 non-null  object 
 3   drivetrain           10000 non-null  object 
 4   num_doors            10000 non-null  int64  
 5   engine_displacement  10000 non-null  int64  
 6   num_cylinders        10000 non-null  int64  
 7   horsepower           9123 non-null   float64
 8   vehicle_weight       10000 non-null  int64  
 9   acceleration         9736 non-null   float64
 10  fuel_efficiency_mpg  10000 non-null  float64
dtypes: float64(3), int64(5), object(3)
memory usage: 859.5+ KB


### Q2. Records count

How many records (rows) are in the dataset?

**Concept:** a DataFrame has a shape — `(rows, columns)`.

**Tools:** `df.shape` or `len(df)`

In [47]:
df.shape


(10000, 11)

### Q3. Fuel types

How many fuel types are in the dataset?

**Concept:** "how many *distinct* values" is different from "how many rows".
A categorical column has a small set of unique values repeated many times.

**Tools:** `df['fuel_type'].nunique()` counts them.
`df['fuel_type'].unique()` shows what they actually are — run that too, so you see
the data instead of just a number.

In [48]:
df["fuel_type"].nunique()
df["fuel_type"].unique()

array(['Gasoline', 'Diesel', 'Hybrid'], dtype=object)

### Q4. Missing values

How many **columns** have missing values?

**Concept:** this is a two-step reduction, and it's worth understanding the shape at each step:
1. `df.isnull()` → a DataFrame of True/False, same size as `df`
2. `.sum()` → sums down each column, giving one number **per column** (True counts as 1)
3. then count how many of those numbers are greater than zero

**Careful:** the question asks how many *columns* have missing values, not how many
missing values there are in total. Read the number you produce and check it answers
the question that was asked. This distinction catches a lot of people.

In [49]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

### Q5. Max fuel efficiency of cars from Asia

**Concept: boolean masking** — the core pandas idea. `df['origin'] == 'Asia'` produces a
Series of True/False, one per row. Passing that back into `df[...]` keeps only the True rows.
Almost everything you do in pandas for the rest of this course is a variation of this.

**Tools:** `df[df['origin'] == 'Asia']` then `['fuel_efficiency_mpg'].max()`

Check the spelling of the origin values first with `df['origin'].unique()` — a silent
typo gives you an empty selection and a confusing `NaN`.

In [51]:
asia = df[df["origin"] == "Asia"]
print(len(asia), asia["origin"].unique())   
asia["fuel_efficiency_mpg"].max()

2994 ['Asia']


np.float64(41.2)

### Q6. Median value of horsepower

Four steps:
1. median of `horsepower`
2. the **most frequent** value of `horsepower`
3. `fillna` the missing values with that most frequent value
4. median again — did it change?

**Concept:** this is your first taste of *imputation* — filling missing values instead of
dropping the rows. The point of the question is to notice that **how you fill missing data
changes your statistics.** That is a real modelling decision, not a formality.

**Tools:** `.median()`, `.mode()`, `.fillna()`

**Two traps:**
- `.mode()` returns a *Series*, not a single number (there can be ties). Take `[0]`.
- `.fillna()` returns a **new** Series by default; it does not modify `df` in place.
  Assign the result to something.

Think about *why* the median moved the way it did (or didn't) before you pick an option.

In [52]:
before     = df["horsepower"].median()
fill_value = df["horsepower"].mode()[0]     
filled     = df["horsepower"].fillna(fill_value)   
after      = filled.median()

print(before, fill_value, after)

254.0 252.0 252.0


### Q7. Sum of weights

Follow the 9 steps in the homework. Tools you need, in order:

| step | tool |
|---|---|
| select Asia rows | boolean mask (same as Q5) |
| select 2 columns | `df[['vehicle_weight', 'model_year']]` — note the **double** brackets |
| first 7 rows | `.head(7)` |
| to NumPy array | `.values` (or `.to_numpy()`) |
| transpose | `X.T` |
| matrix multiply | `X.T @ X` (the `@` operator) |
| invert | `np.linalg.inv(...)` |
| sum | `.sum()` |

**This is the most important cell in the whole homework.** What you are computing is:

$$w = (X^T X)^{-1} X^T y$$

That is the **normal equation** — the closed-form solution for linear regression. You are
not doing busywork with matrices; you are training a linear regression model by hand, with
no library. `X` holds the features, `y` holds the targets, and `w` comes out as the weights
the model learned.

Module 2 will spend a whole week on this formula. Get familiar with it here and Module 2
becomes easy instead of mysterious.

Check the shape of every intermediate result (`X.shape`, `XTX.shape`) — in linear algebra,
shape mismatches are where all the bugs live.

In [ ]:
X = asia[["vehicle_weight", "model_year"]].head(7).values
print(X.shape)           

XTX = X.T @ X             
XTX_inv = np.linalg.inv(XTX)

y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
w = XTX_inv @ X.T @ y

print(w, w.sum())

(7, 2)
[0.13644777 0.2327492 ] 0.36919696904925486


---

## Before you submit

For each question, note the option from the homework that matches your number.
If a computed value doesn't match **any** option, don't force it — that means a step is
wrong, and finding out which one is the actual learning.

Then: **Learning in Public.** It gives bonus points and it is genuinely the highest-value
part of the homework — writing "what I struggled with" out loud is what turns watching
videos into understanding. Templates are in the homework file. Tag #mlzoomcamp.